# 03 · Cell type annotation with HeteroSC

Inputs: the embeddings from notebooks `02a–d` (`outputs/embeddings/{dataset}/{expert}/…h5ad`) and the shared master split.
Steps: align the four experts on common cells → L2-normalise → train HeteroSC (adaptive) → train the **uniform-fusion control**
and **single-expert baselines** under identical settings → fine-tuned and zero-shot kNN evaluation → routing analysis.

Protocol (manuscript Section 2.3): class-weighted cross-entropy with label smoothing + diversity regulariser, Accuracy and Macro-F1 on
held-out test cells, zero-shot kNN = 5 neighbours / cosine / distance-weighted voting.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt

from heterosc import (AnnotationConfig, AnnotationTrainer, HeteroSCClassifier, load_dataset, make_loaders,
                      knn_probe, l2_normalize, normalized_entropy)
from heterosc.plots import plot_routing_by_group

EMB_ROOT = Path("outputs/embeddings")
DATASET = "Myeloid"
EXPERTS = ["geneformer", "scgpt", "cellfm", "scfoundation"]
OUT = Path("outputs/annotation") / DATASET

In [ ]:
emb, y, split, classes = load_dataset(EMB_ROOT, DATASET, EXPERTS)      # keeps only cells present in all four experts
emb = {k: l2_normalize(v) for k, v in emb.items()}
print(f"{len(y)} cells | {len(classes)} cell types | " + str(pd.Series(split).value_counts().to_dict()))
print({k: v.shape[1] for k, v in emb.items()})

## Train one model: helper that also reports the zero-shot kNN result

In [ ]:
def fit(experts, fusion, name, epochs=150):
    sub = {k: emb[k] for k in experts}
    cfg = AnnotationConfig(expert_dims={k: v.shape[1] for k, v in sub.items()}, num_classes=len(classes), fusion=fusion, epochs=epochs)
    train, val, test = make_loaders(sub, y, split, cfg.batch_size)
    model = HeteroSCClassifier(cfg)
    res = AnnotationTrainer(model, cfg, OUT / name).fit(train, val, test, y[split == "train"], log_every=25)
    fused = lambda s: model.fused_embeddings({k: torch.tensor(v[split == s]).to(cfg.device) for k, v in sub.items()}).cpu().numpy()
    knn = knn_probe(fused("train"), y[split == "train"], fused("test"), y[split == "test"])
    return {"name": name, "res": res, "knn": knn}

runs = [fit(EXPERTS, "adaptive", "heterosc_adaptive"), fit(EXPERTS, "uniform", "heterosc_uniform")]
runs += [fit([e], "uniform", f"single_{e}") for e in EXPERTS]          # one expert = adapter + head only

In [ ]:
table = pd.DataFrame([{
    "model": r["name"], "FT accuracy": r["res"]["test"]["accuracy"], "FT macro-F1": r["res"]["test"]["macro_f1"],
    "kNN accuracy": r["knn"]["accuracy"], "kNN macro-F1": r["knn"]["macro_f1"]} for r in runs]).set_index("model").round(4)
table

## Routing analysis
Mean routing weight per expert for the adaptive model, overall and per cell type. The overall normalised entropy (Eq. 9) says how
evenly the experts are used; the per-cell-type view says whether different cell types lean on different experts.

In [ ]:
ad_run = runs[0]["res"]["test"]
alphas, labels = ad_run["alphas"], ad_run["labels"]
print("mean routing weights:", dict(zip(EXPERTS, np.round(alphas.mean(0), 3))), "| normalised entropy:", round(normalized_entropy(alphas.mean(0)), 3))
plot_routing_by_group(alphas, labels, EXPERTS, group_names=[classes[i] for i in sorted(set(labels))], title=f"{DATASET}: routing weight per cell type")
plt.tight_layout(); plt.show()

Interpret routing weights as *predictive contributions under this objective*, not as intrinsic quality of a model: the diversity
regulariser favours balanced use, and weights are not evidence of biological specialisation.